# core
> VPS provisioning: cloud-init generation, Hetzner hcloud CLI wrapper, SSH deployment helpers

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, json, subprocess, time
from dockeasy import Cli
from fastcore.all import L, Path, run, listify, atomic_save, AttrDict
from fastcloudinit.core import cloud_init_base, cloud_init_config, user, runcmd, reboot
from hcloud import Client
from hcloud.images import Image
from hcloud.server_types import ServerType
from hcloud.locations import Location
from hcloud.ssh_keys import SSHKey

## Multipass for local testing
> Using `multipass` to test cloud-init and deployment locally before provisioning real VPSes. You'll need to install Multipass and have it in your PATH for this to work.

In [ ]:
#| export
class Multipass(Cli):
    'Wrap multipass CLI: manage local Ubuntu VMs for testing'
    def _run(self, cmd, *args): return run('multipass', cmd, *args)
    def launch(self,name,image='24.04',cpus=1,memory='1G',disk='10G',cloud_init:AttrDict=None, mounts=None) -> AttrDict:
        'Launch a VM. cloud_init: AttrDict(yaml, key) from multi_init() or vps_init(). Returns AttrDict(name, key).'
        args = ['multipass', 'launch', image, '-n', name, '-c', str(cpus), '-m', memory, '-d', disk]
        for hp, vp in (mounts or {}).items(): args += ['--mount', f'{hp}:{vp}']
        if cloud_init: args += ['--cloud-init', '-']
        subprocess.run(args, input=cloud_init.yaml if cloud_init else None, text=True, check=True)
        return AttrDict(name=name, key=cloud_init.key if cloud_init else None)

    def vms(self, running=False):
        'List VM names. running=True filters to Running state.'
        lst = L(json.loads(self('list', '--format', 'json')).get('list', []))
        if running: lst = lst.filter(lambda v: v.get('state') == 'Running')
        return list(lst.itemgot('name'))

    def ip(self, name) -> str:
        'Get IPv4 address of a VM.'
        return json.loads(self('info', name, '--format', 'json'))['info'][name]['ipv4'][0]

    def exec_(self, name, *cmd) -> str:
        'Run a command in a VM.'
        return self('exec', name, '--', *cmd)

    def rm(self, name, purge=True) -> None:
        'Delete a VM.'
        self('delete', '--purge' if purge else '', name)

    def transfer(self, src, dst) -> None:
        'Transfer files to/from a VM. Use "vmname:/path" for VM paths.'
        self('transfer', src, dst)

mp = Multipass()

def deploy_mp(name, src, path='/srv/app', build=True) -> str:
    'Sync local directory into a Multipass VM and run docker compose up -d.'
    mp.exec_(name, 'mkdir', '-p', path)
    mp.transfer(str(src).rstrip('/') + '/', f'{name}:{path}')
    cmd = ['docker', 'compose', '-f', f'{path}/docker-compose.yml', 'up', '-d', '--remove-orphans']
    if build: cmd.append('--build')
    return mp.exec_(name, *cmd)

In [ ]:
Multipass().vms()

['testvm']

## Cloud-init generation

`vps_init()` builds a cloud-init YAML for a fresh VPS via `fastcloudinit.cloud_init_config`: UFW hardening, user creation, SSH key setup, optional Docker. `multi_init()` is the local equivalent via `cloud_init_base` — same Docker setup, no UFW, no fail2ban.

In [ ]:
#| export
dock_cmd = [
    'curl -fsSL https://get.docker.com | sh',
    'usermod -aG docker {username}',
    'systemctl enable --now docker',
]

def gen_key(slug, key_dir=None):
    'Generate ed25519 key pair at <key_dir>/<slug>. Overwrites existing. Returns AttrDict(key, pub, pub_str).'
    d = Path(key_dir or Path.home()/'.ssh')
    priv, pub = d/slug, d/f'{slug}.pub'
    priv.unlink(missing_ok=True); pub.unlink(missing_ok=True)
    subprocess.run(['ssh-keygen', '-t', 'ed25519', '-f', str(priv), '-N', ''], check=True, capture_output=True)
    return AttrDict(key=priv, pub=pub, pub_str=pub.read_text().strip())

def vps_init(hostname, pub_keys=None, username='deploy', docker=True, pkgs=None, cmds=None, **kw):
    'Cloud-init for a fresh VPS. pub_keys=None → auto-generates ed25519 key pair. Returns AttrDict(yaml, key).'
    kp = gen_key(hostname) if pub_keys is None else None
    pks = [kp.pub_str] if kp else listify(pub_keys)
    rcmds = list(cmds or [])
    if docker: rcmds = [c.format(username=username) for c in dock_cmd] + rcmds
    yaml = cloud_init_config(hostname=hostname, username=username, pub_keys=pks,
        packages=['curl', 'fail2ban', 'unattended-upgrades'] + list(pkgs or []), cmds=rcmds, **kw)
    return AttrDict(yaml=yaml, key=kp.key if kp else None)

In [ ]:
#| export
def multi_init(hostname, pub_keys=None, username='deploy', docker=True, pkgs=None, cmds=None):
    'Cloud-init for Multipass local VMs. pub_keys=None → auto-generates ed25519 key pair. Returns AttrDict(yaml, key).'
    kp = gen_key(hostname) if pub_keys is None else None
    pks = [kp.pub_str] if kp else listify(pub_keys)
    rcmds = list(cmds or [])
    if docker: rcmds = [c.format(username=username) for c in dock_cmd] + rcmds
    kw = {**runcmd(rcmds), **reboot()} if rcmds else {}
    yaml = cloud_init_base(hostname, packages=['curl'] + list(pkgs or []), users=[user(username, pks)], **kw)
    return AttrDict(yaml=yaml, key=kp.key if kp else None)

In [ ]:
# explicit pub_keys → no key generated, key=None
ci = multi_init('mylocal', 'ssh-rsa AAAA...')
assert '#cloud-config' in ci.yaml and 'get.docker.com' in ci.yaml and 'ufw' not in ci.yaml
assert 'power_state' in ci.yaml and ci.key is None
print(ci.yaml)

# auto-generate mode: key pair written to ~/.ssh/mylocal{,.pub}
ci2 = multi_init('mylocal', docker=False)
assert 'get.docker.com' not in ci2.yaml and 'power_state' not in ci2.yaml
assert ci2.key is not None and ci2.key.exists()
assert Path(str(ci2.key) + '.pub').exists()
ci2.key.unlink(); Path(str(ci2.key) + '.pub').unlink()
print('multi_init OK')

#cloud-config
hostname: mylocal
preserve_hostname: false
packages:
- curl
package_update: true
package_upgrade: true
disable_root: true
ssh_pwauth: false
users:
- name: deploy
  groups:
  - sudo
  shell: /bin/bash
  sudo:
  - ALL=(ALL) NOPASSWD:ALL
  ssh_authorized_keys:
  - ssh-rsa AAAA...
runcmd:
- curl -fsSL https://get.docker.com | sh
- usermod -aG docker deploy
- systemctl enable --now docker
power_state:
  mode: reboot
  message: Rebooting
  timeout: 1
  condition: true

multi_init OK


In [ ]:
ci = vps_init('myserver', 'ssh-rsa AAAA...', docker=True)
assert '#cloud-config' in ci.yaml and 'get.docker.com' in ci.yaml
assert 'fail2ban' in ci.yaml and 'unattended-upgrades' in ci.yaml
assert ci.key is None  # explicit pub_keys → no key generated
print(ci.yaml)
print('vps_init OK')

#cloud-config
hostname: myserver
preserve_hostname: false
packages:
- curl
- fail2ban
- unattended-upgrades
package_update: true
package_upgrade: true
disable_root: true
ssh_pwauth: false
users:
- name: deploy
  groups:
  - sudo
  shell: /bin/bash
  sudo:
  - ALL=(ALL) NOPASSWD:ALL
  ssh_authorized_keys:
  - ssh-rsa AAAA...
runcmd:
- curl -fsSL https://get.docker.com | sh
- usermod -aG docker deploy
- systemctl enable --now docker
- ufw default deny incoming
- ufw default allow outgoing
- ufw logging off
- ufw allow 22/tcp
- ufw --force enable
apt:
  conf: 'APT::Periodic::Update-Package-Lists "1";

    APT::Periodic::Download-Upgradeable-Packages "1";

    APT::Periodic::AutocleanInterval "7";

    APT::Periodic::Unattended-Upgrade "0";

    Unattended-Upgrade::Automatic-Reboot "false";

    '
write_files:
- path: /etc/logrotate.d/00-cloud-init-global
  owner: root:root
  permissions: '0644'
  content: "/var/log/*.log {\n    weekly\n    rotate 7\n    compress\n    su root adm\n    crea

## Hetzner (hcloud Python SDK)

`Hetzner` wraps the hcloud Python SDK — no CLI binary or config files required. Token read from `HCLOUD_TOKEN` by default.

In [ ]:
#| export
class Hetzner:
    'Hetzner Cloud VPS provider via hcloud Python SDK. Token read from HCLOUD_TOKEN by default.'
    def __init__(self, token=None):
        token = token or os.environ.get('HCLOUD_TOKEN')
        if not token: raise ValueError('HCLOUD_TOKEN environment variable not set')
        self._c = Client(token=token)

    def servers(self) -> list:
        'List servers as [{name, ip, status}]'
        return L(self._c.servers.get_all()).map(lambda s: dict(name=s.name, ip=s.public_net.ipv4.ip, status=s.status))

    def server_ip(self, name) -> str:
        'Get public IPv4 of a server by name'
        s = self._c.servers.get_by_name(name)
        if not s: raise ValueError(f'Server {name!r} not found')
        return s.public_net.ipv4.ip

    def create(self, name, image='ubuntu-24.04', server_type='cx23', location=None, cloud_init=None, ssh_keys=None):
        'Create a server. cloud_init: YAML string or AttrDict(yaml, key) from vps_init(). Returns AttrDict(ip, name, key, resp).'
        key = None
        if isinstance(cloud_init, AttrDict): key, cloud_init = cloud_init.key, cloud_init.yaml
        resp = self._c.servers.create(
            name=name,
            server_type=ServerType(name=server_type),
            image=Image(name=image),
            location=Location(name=location) if location else None,
            user_data=cloud_init,
            ssh_keys=[SSHKey(name=k) for k in (ssh_keys or [])],
        )
        return AttrDict(ip=resp.server.public_net.ipv4.ip, name=name, key=key, resp=resp)

    def delete(self, name) -> None:
        'Delete a server by name'
        s = self._c.servers.get_by_name(name)
        if s: s.delete()

    def keys(self) -> list:
        'List SSH keys as [{name, fingerprint}]'
        return L(self._c.ssh_keys.get_all()).map(lambda k: dict(name=k.name, fingerprint=k.fingerprint))

    def key_names(self) -> list:
        'Return SSH key name strings for use in create(ssh_keys=[...])'
        return [k['name'] for k in self.keys()]

### Integration tests

Requires `HCLOUD_TOKEN` in the environment. Creates a minimal `cx11` server, verifies the lifecycle, then deletes it.

In [ ]:
#| eval: False
hz = Hetzner()
hz.servers()

[{'name': 'vedicreader-cx32-hel', 'ip': '46.62.133.112', 'status': 'running'}]

In [ ]:
#| eval: False
_TOKEN = os.environ.get('HCLOUD_TOKEN')
_TEST_SERVER = 'fastops-test'

if not _TOKEN: print('HCLOUD_TOKEN not set — skipping hcloud integration tests')
else:
    hz = Hetzner()
    hz.delete(_TEST_SERVER)

    svr = hz.create(_TEST_SERVER, server_type='cx23', location='hel1')
    assert svr.ip, 'create() should return an IP'
    print(f'create OK: {svr.ip}')

    svrs = hz.servers()
    assert _TEST_SERVER in [s['name'] for s in svrs]
    print(f'servers() OK: {[s["name"] for s in svrs]}')

    assert hz.server_ip(_TEST_SERVER) == svr.ip
    print(f'server_ip() OK: {svr.ip}')
    print(f'key_names() OK: {hz.key_names()}')

    hz.delete(_TEST_SERVER)
    assert _TEST_SERVER not in [s['name'] for s in hz.servers()]
    print('delete() OK\nAll hcloud tests passed!')

create OK: 77.42.122.33
servers() OK: ['vedicreader-cx32-hel', 'fastops-test']
server_ip() OK: 77.42.122.33
key_names() OK: []
delete() OK
All hcloud tests passed!


## SSH helpers

Pure subprocess-based SSH/rsync utilities — no paramiko dependency. `deploy()` syncs a Compose stack to a remote host and brings it up.

In [ ]:
#| export
def _ssh_base(host, user, key, port):
    a = ['ssh', '-o', 'StrictHostKeyChecking=accept-new']
    if key: a += ['-i', str(key)]
    if port != 22: a += ['-p', str(port)]
    return a + [f'{user}@{host}']

def _resolve_key(key=None, name=None):
    'Resolve SSH key: explicit path > name slug (~/.ssh/<name>) > SSH_KEY_PATH env > SSH_PRIVATE_KEY env.'
    if key: return str(key)
    if name:
        p = Path.home()/'.ssh'/name
        if not p.exists(): raise FileNotFoundError(f'No SSH key at {p} — run gen_key({name!r}) first')
        return str(p)
    if p := os.environ.get('SSH_KEY_PATH'): return p
    pk = os.environ.get('SSH_PRIVATE_KEY')
    if not pk: return None
    if not (p := Path(f'/tmp/.vpseasy_{os.getuid()}.pem')).exists():
        with atomic_save(p, mode='w') as f: f.write(pk.strip() + '\n')
        os.chmod(p, 0o400)
    return str(p)

def run_ssh(host, *cmds, user='deploy', key=None, name=None, port=22, capture=False, check=True):
    'Run commands on remote host via SSH. capture=True returns stdout string.'
    r = subprocess.run(_ssh_base(host, user, _resolve_key(key, name), port) + [' && '.join(cmds)],
                       capture_output=capture, text=capture, check=check)
    return r.stdout if capture else r

def sync(host, src='.', path='/srv/app', user='deploy', key=None, name=None, include=None, exclude=None):
    'Rsync local src to remote host:path. include= whitelist patterns, exclude= blacklist patterns.'
    run_ssh(host, f'mkdir -p {path}', user=user, key=key, name=name)
    ssh_e = ' '.join(_ssh_base(host, user, _resolve_key(key, name), 22)[:-1])
    inc, exc = listify(include), listify(exclude)
    cmd = ['rsync', '-az' + ('m' if inc else ''), '--delete', '-e', ssh_e]
    for p in exc: cmd += ['--exclude', p]
    if inc:
        for p in inc: cmd += (['--include', p] if not p.endswith('/') else
                               ['--include', p.rstrip('/'), '--include', p + '**'])
        cmd += ['--include', '*/', '--exclude', '*']
    cmd += [str(src).rstrip('/') + '/', f'{user}@{host}:{path}/']
    subprocess.run(cmd, check=True)

def deploy(host, src='.', path='/srv/app', user='deploy', build=True, key=None, name=None,
           include=None, exclude=None):
    'Sync src to host via rsync then docker compose up.'
    sync(host, src, path, user, key=key, name=name, include=include, exclude=exclude)
    run_ssh(host, f'cd {path} && docker compose up -d --remove-orphans' + (' --build' if build else ''),
            user=user, key=key, name=name)

In [ ]:
import tempfile
# _ssh_base: pure flag construction
assert _ssh_base('1.2.3.4', 'deploy', None, 22) == ['ssh', '-o', 'StrictHostKeyChecking=accept-new', 'deploy@1.2.3.4']
assert _ssh_base('1.2.3.4', 'deploy', '/k', 2222)[-3:] == ['-p', '2222', 'deploy@1.2.3.4']
assert '-i' in _ssh_base('h', 'u', '/my/key', 22)
print('_ssh_base OK')

# _resolve_key: explicit path, name slug, SSH_KEY_PATH env, SSH_PRIVATE_KEY env
assert _resolve_key('/tmp/mykey') == '/tmp/mykey'
_saved = {k: os.environ.pop(k, None) for k in ('SSH_KEY_PATH', 'SSH_PRIVATE_KEY')}

# name slug: found → returns path; missing → raises
_kf = Path.home()/'.ssh'/'_vpseasy_test_slug'
_kf.write_text('FAKE'); assert _resolve_key(name='_vpseasy_test_slug') == str(_kf); _kf.unlink()
try: _resolve_key(name='_no_such_key_xyz'); assert False
except FileNotFoundError: pass

os.environ['SSH_KEY_PATH'] = '/tmp/via-env'
assert _resolve_key() == '/tmp/via-env'
del os.environ['SSH_KEY_PATH']

_expected = f'/tmp/.vpseasy_{os.getuid()}.pem'
Path(_expected).unlink(missing_ok=True)
os.environ['SSH_PRIVATE_KEY'] = 'FAKE-KEY-CONTENT'
k = _resolve_key()
assert k == _expected and Path(k).exists() and Path(k).stat().st_mode & 0o777 == 0o400
assert _resolve_key() == k  # idempotent
Path(k).unlink(); del os.environ['SSH_PRIVATE_KEY']
for k, v in _saved.items():
    if v: os.environ[k] = v
assert _resolve_key() is None
print('_resolve_key OK')

_ssh_base OK
_resolve_key OK


## Verification helpers

Cloud-init runs asynchronously after `create()` returns. Use `wait_ssh()` to block until the server is reachable, then `check_cloud_init()` and `check_docker()` to confirm the bootstrap completed successfully before deploying.

In [ ]:
#| export
def wait_ssh(host, u='deploy', k=None, p=22, tout=120, interval=5):
    'Poll SSH until connection succeeds or raises TimeoutError.'
    dl = time.time() + tout
    while time.time() < dl:
        try: run_ssh(host, 'true', user=u, key=k, port=p); return True
        except: time.sleep(interval)
    raise TimeoutError(f'SSH to {host} not ready after {tout}s')

def chk_cloud_init(host, u='deploy', k=None) -> str:
    'Return cloud-init status: done|running|error|unknown'
    try:
        o = run_ssh(host, 'sudo cloud-init status', user=u, key=k, capture=True)
        return o.split(': ', 1)[-1].strip() if ': ' in o else o.strip()
    except Exception: return 'unknown'

def chk_docker(host, u='deploy', k=None) -> bool:
    'Verify docker daemon is running and user can run containers.'
    try: run_ssh(host, 'docker info', user=u, key=k); return True
    except Exception: return False

## SSH key helpers

`load_pub_keys()` resolves a list of paths (or auto-detects `~/.ssh/id_*.pub`) into a flat list of public key strings ready for `vps_init()` and `multi_init()`.

In [ ]:
#| export
def load_pub_keys(paths=None) -> list:
    'Load SSH public key strings. paths=None → auto-detect from ~/.ssh/id_*.pub'
    paths = paths or list(Path.home().glob('.ssh/id_*.pub'))
    return [Path(p).read_text().strip() for p in listify(paths) if Path(p).exists()]

In [ ]:
import tempfile

In [ ]:
keys = load_pub_keys()
print(f'Found {len(keys)} local SSH key(s)')

_f = Path(tempfile.mktemp(suffix='.pub'))
_f.write_text('ssh-ed25519 TESTKEY comment')
assert load_pub_keys([str(_f)]) == ['ssh-ed25519 TESTKEY comment']
_f.unlink()
assert load_pub_keys(['/no_such_key.pub']) == []
print('load_pub_keys OK')

# gen_key: creates ed25519 pair, returns AttrDict, overwrites existing
_d = Path(tempfile.mkdtemp())
kp = gen_key('testkey', key_dir=_d)
assert kp.key.exists() and kp.pub.exists()
assert kp.pub_str.startswith('ssh-ed25519')
kp2 = gen_key('testkey', key_dir=_d)  # overwrite — should not raise
assert kp2.pub_str.startswith('ssh-ed25519')
import shutil; shutil.rmtree(_d)
print('gen_key OK')

Found 4 local SSH key(s)
load_pub_keys OK
gen_key OK


## Integration tests: SSH helpers and verification helpers

Requires Multipass. Launches a minimal Ubuntu VM with local SSH keys injected via `multi_init`, then exercises `wait_ssh`, `chk_cloud_init`, `chk_docker`, `run_ssh`, and `sync`.

In [ ]:
#| eval: False
# --- setup: auto-generate key pair, inject via multi_init, launch VM ---
_VM = 'testvm'
mp = Multipass()
try: mp.rm(_VM)
except: pass

ci = multi_init(_VM, docker=False) # generates ~/.ssh/fastops-vpstest{,.pub}
vm = mp.launch(_VM, image='24.04', cpus=1, memory='512M', disk='5G', cloud_init=ci)
ip = mp.ip(vm.name)
_key = vm.key
print(f'VM at {ip}, key: {_key}')

Creating testvm  Configuring testvm  Starting testvm  Waiting for initialization to complete  Launched: testvm
VM at 192.168.2.25, key: /Users/71293/.ssh/testvm


In [ ]:
#| eval: False
# wait_ssh / chk_cloud_init / run_ssh
assert wait_ssh(ip, k=_key, tout=30) is True; print('wait_ssh OK')
status = chk_cloud_init(ip, k=_key)
assert status in ('done', 'running'), f'unexpected: {status!r}'; print(f'chk_cloud_init: {status}')
assert run_ssh(ip, 'echo hi', key=_key, capture=True).strip() == 'hi'; print('run_ssh OK')

wait_ssh OK
chk_cloud_init: done
run_ssh OK


In [ ]:
#| eval: False
# sync: full dir / exclude / include (whitelist)
import tempfile
_src = Path(tempfile.mkdtemp())
(_src/'a.txt').write_text('hello')
(_src/'b.log').write_text('log')
(_src/'sub').mkdir(); (_src/'sub'/'c.txt').write_text('sub')

sync(ip, _src, '/tmp/t1', key=_key)
assert run_ssh(ip, 'cat /tmp/t1/a.txt', key=_key, capture=True).strip() == 'hello'; print('sync full OK')

sync(ip, _src, '/tmp/t2', key=_key, exclude=['*.log'])
assert run_ssh(ip, 'ls /tmp/t2', key=_key, capture=True).split() == ['a.txt', 'sub']; print('sync exclude OK')

sync(ip, _src, '/tmp/t3', key=_key, include=['a.txt'])
assert run_ssh(ip, 'ls /tmp/t3', key=_key, capture=True).strip() == 'a.txt'; print('sync include OK')

sync(ip, _src, '/tmp/t4', key=_key, include=['sub/'])
assert run_ssh(ip, 'cat /tmp/t4/sub/c.txt', key=_key, capture=True).strip() == 'sub'; print('sync include-dir OK')

sync full OK
sync exclude OK
sync include OK
sync include-dir OK


In [ ]:
#| eval: False
# --- teardown ---
mp.rm(_VM); print('VM removed')

VM removed


In [ ]:
#| export
def mv_skill_md(dry_run=True, dir=None) -> None:
    'Copy bundled SKILL.md to .agents/skills/vpseasy/ and ~/.claude/skills/vpseasy/'
    base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    src = base/'SKILL.md'
    if not src.exists(): return
    root = Path(dir or '.')
    ts = [root/'.agents/skills/vpseasy/SKILL.md',
          Path.home()/'.claude/skills/vpseasy/SKILL.md']
    if dry_run: print(f'Would copy to: {[str(p) for p in ts]}')
    else: [p.mk_write(src.read_text(encoding='utf-8')) for p in ts]
    if not dry_run: print(f'Installed → {[str(p) for p in ts]}')

In [ ]:
import io, sys, tempfile, os
_d = Path(tempfile.mkdtemp())
(_d/'SKILL.md').write_text('test-skill')   # provide a src so function doesn't bail early
_saved_cwd = os.getcwd(); os.chdir(_d)
_buf = io.StringIO(); sys.stdout, _saved = _buf, sys.stdout
mv_skill_md(dry_run=True, dir=str(_d))
sys.stdout = _saved; os.chdir(_saved_cwd)
_out = _buf.getvalue()
assert '.agents/skills/vpseasy/SKILL.md' in _out, _out
assert '.claude/skills/vpseasy/SKILL.md' in _out, _out
print('mv_skill_md dry_run OK')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()